# Opportunity · Concentration & Rebalancing

A real portfolio drifts toward its winners. This puts numbers on the drift against a
target *you* set, shows the trades that close the gap, and ranks holdings by **risk
reduced per dollar trimmed** — because not every dollar of concentration costs the same
amount of volatility.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from invest import config, analysis as A, ledger, mapping, opportunities as O

px.defaults.template = "plotly_white"
PALETTE = px.colors.qualitative.Safe

positions = pd.read_parquet(config.POSITIONS_PARQUET)
history = pd.read_parquet(config.PRICES_PARQUET)
transactions = pd.read_parquet(config.TRANSACTIONS_PARQUET)
entries, _, _ = ledger.load()
account_meta = mapping.load_account_meta()

TODAY = pd.Timestamp.today().normalize()
TOTAL = positions["market_value"].sum()
print(f"${TOTAL:,.0f} · {positions['account_name'].nunique()} accounts · as of {TODAY:%Y-%m-%d}")

## Set your target mix

Edit `TARGETS` (asset-class weights, as fractions — they need not sum to 1; the rest is
treated as "everything else"). The defaults below are illustrative, not a recommendation.

In [ ]:
TARGETS = {
    "target_date": 0.25,
    "us_equity": 0.25,
    "employer_stock": 0.05,   # cap single-employer risk
    "cash": 0.10,
    "bdc": 0.04,
}
by_class = positions.groupby("asset_class")["market_value"].sum()
weights = by_class / by_class.sum()
plan = O.rebalance_plan(weights, TARGETS, total=TOTAL)
display(plan.style.format({"current": "{:.1%}", "target": "{:.1%}",
                           "drift": "{:+.1%}", "trade": "${:+,.0f}"}))

## Current vs target, and the trades to get there

Left: where each sleeve stands against target. Right: the dollar trade to close it
(green = add, red = trim).

In [ ]:
show = plan[plan["drift"].abs() > 0.005]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Current vs target", "Rebalancing trade ($)"),
                    column_widths=[0.55, 0.45])
fig.add_bar(y=show.index, x=show["current"], name="current", orientation="h",
            marker_color=PALETTE[0], row=1, col=1)
fig.add_bar(y=show.index, x=show["target"], name="target", orientation="h",
            marker_color="rgba(0,0,0,0.25)", row=1, col=1)
fig.add_bar(y=show.index, x=show["trade"], orientation="h", showlegend=False,
            marker_color=["#27ae60" if v > 0 else "#c0392b" for v in show["trade"]], row=1, col=2)
fig.update_xaxes(tickformat=".0%", row=1, col=1); fig.update_xaxes(tickprefix="$", row=1, col=2)
fig.update_layout(height=440, barmode="group", title="Rebalancing plan")
fig.show()

## Risk-reduction trim guide

Each holding's share of **portfolio variance** vs its capital weight. Names where risk
towers over weight (top of the chart) cut the most volatility per dollar sold — the
highest-leverage trims if de-risking is the goal.

In [ ]:
ret = A.holding_returns(positions, history)
rr = O.risk_reduction_ranking(A.portfolio_weights(positions), ret)
fig = go.Figure()
fig.add_bar(x=rr.index, y=rr["weight"], name="capital weight", marker_color=PALETTE[2])
fig.add_bar(x=rr.index, y=rr["risk_contribution"], name="risk share", marker_color=PALETTE[3])
fig.update_layout(barmode="group", height=420, yaxis_tickformat=".0%",
                  title="Risk share vs weight — trim where risk ≫ weight", hovermode="x unified")
fig.show()
top = rr.index[0]
print(f"Highest-leverage trim: {top} — {rr.loc[top,'risk_contribution']:.0%} of portfolio "
      f"risk at {rr.loc[top,'weight']:.0%} of capital "
      f"({rr.loc[top,'risk_per_weight']:.1f}× its weight).")

## Single-name & employer concentration

Position sizes against a single-name cap you set. Employer stock (here MSFT) stacks
*investment* risk on top of *income* risk — same source for your paycheck and a chunk of
your net worth.

In [ ]:
CAP = 0.10
sym_w = A.portfolio_weights(positions, include_cash=True)
big = sym_w[sym_w > CAP]
cls = positions.groupby("symbol")["asset_class"].first()
fig = px.bar(x=big.index, y=big.values,
             color=[("employer_stock" if cls.get(s) == "employer_stock" else "holding") for s in big.index],
             color_discrete_map={"employer_stock": "#c0392b", "holding": PALETTE[0]},
             title=f"Holdings above the {CAP:.0%} single-name cap")
fig.add_hline(y=CAP, line_dash="dot", line_color="grey", annotation_text=f"{CAP:.0%} cap")
fig.update_layout(height=400, yaxis_tickformat=".0%", xaxis_title="", yaxis_title="weight",
                  showlegend=True, legend_title="")
fig.show()
emp = positions[positions["asset_class"] == "employer_stock"]["market_value"].sum()
if emp:
    print(f"Employer stock (MSFT): ${emp:,.0f} = {emp/TOTAL:.1%} of the portfolio — "
          f"concentration plus correlation with your income.")

---
*These are **candidate generators**, not advice — every row is a starting point for your
own judgment (and, for anything tax-related, a CPA's). Prices are research-grade
(yfinance); cost basis and lots come from the curated ledger, so they're only as right as
its fixups.*